## **Курсовая работа 3-го курса ОП "Компьютерные науки и технологии" в НИУ ВШЭ-НН**
### **Название: Мониторинг репутации компаний через анализ упоминаний в сети**

**Срок выполнения:** с ..2026 г. по 21.05.2026 г.  
**Выполнил данную часть:** Иван Гайфиев (23КНТ-6)


### **Описание задания:**
Требуется разработать набор алгоритмов, задачей которых является NLP-анализ входных данных (упоминаний о компаниях) с целью формулировки основных тем, выявления наиболее часто упоминаемых слов и определения лучших и худших сторон компании на основе отзывов.

#### **Основные этапы:**
1) Подключение к базе данных Supabase и загрузка из неё набор упоминаний для последующего анализа в формате csv
2) Предобработать датафрейм, оставив данные, соответствующие определённым условиям
3) Разработать алгоритм по моделлированию ключевых тем
4) Разработать алгоритм по формированию облака слов
5) Разработать алгоритм по составлению списков позитивных и негативных аспектов
6) Выполнить анализ исходных данных с помощью этих алгоритмов и выгрузить результаты в БД 

#### **Полезные материалы:**
1) Доска MIRO с описанием системы и её компонентов - https://miro.com/app/board/uXjVHW0IYa8=/


### **Выполнение задания:**


#### **Этап 1: Подключение к БД и загрузка данных**
**Задание:**
Подключиться к базе данных Supabase и загрузить из неё набор упоминаний для последующего анализа в формате csv

**Выполнение:**  


In [31]:
! pip install supabase google.colab scikit-learn nltk bertopic sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.6 MB/s eta 0:00:00


In [32]:
# import Topics_analysis as ta # importing file with functions

import os
import pandas as pd
from google.colab import drive
from supabase import create_client, Client
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import nltk
from bertopic import BERTopic

In [36]:
# Loading data
def mount_file():
    drive.mount('/content/drive')
    config_file = "/content/drive/MyDrive/Labs in Colab /Course work 3rd year/config.txt"
    return config_file

def get_config():
    config_file = mount_file()
    with open (config_file, 'r') as f:
        SUPABASE_URL = f.readline().strip()
        SUPABASE_KEY = f.readline().strip()
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    return supabase

def export_data(table_name, company_id, output_file="output.csv"):
    supabase = get_config()
    
    try:
        response = supabase.table(table_name).select("mention_id, text").eq("company_id", company_id).execute()

        if not response.data:
            print(f"Нет данных для company_id={company_id}.")
            return None

        df = pd.DataFrame(response.data)

        df.to_csv(output_file, index=False, encoding="utf-8-sig")

        print(f"Сохранено {len(df)} строк в {output_file}")

        return df
    
    except Exception as e:
        print("Ошибка при получении данных:")
        print(e)
        
df = export_data("mentions", 1)
print(df.head())



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Сохранено 1000 строк в output.csv
   mention_id                                               text
0           1                           реал мешок барса чемпион
1           2  во вконтакте захожу только ради музыки, кино и...
2           3  Я каждый раз выхожу из аккаунта и заново ввожу...
3           4                         не грузит ничего постоянно
4           5  не могу войти в аккаунт, пишет что номер привя...


In [30]:
# TOPIC MODELLING
# Base algorithm

def topics_base(df):
    nltk.download("stopwords")
    russian_stopwords = stopwords.words("russian")
    texts = df["text"].dropna().astype(str).tolist()

    vectorizer = TfidfVectorizer(
        max_features=1000,
        stop_words=russian_stopwords
    )

    X = vectorizer.fit_transform(texts)
    words = vectorizer.get_feature_names_out()

    scores = X.sum(axis=0).A1

    keywords = pd.DataFrame({
        "word": words,
        "score": scores
    }).sort_values("score", ascending=False)

    top_words = keywords.head(20)["word"].tolist()

    topics_df = pd.DataFrame([{
        "method": "TF-IDF",
        "topic": "Общая тема отзывов",
        "keywords": ", ".join(top_words),
        "sentence": f"Основные обсуждаемые темы: {', '.join(top_words[:10])}."
    }])

    return topics_df




In [ ]:
# Middle algorithm

In [ ]:
# Advanced algorithm

def topics_advanced(df):

    docs = df["text"].dropna().astype(str).tolist()

    model = BERTopic(language="multilingual")
    topics, probs = model.fit_transform(docs)

    info = model.get_topic_info()

    topic_rows = []

    for topic_id in info["Topic"]:
        if topic_id == -1:
            continue

        words = model.get_topic(topic_id)
        words = [w for w, _ in words]

        topic_rows.append({
            "method": "BERTopic",
            "topic_id": topic_id,
            "keywords": ", ".join(words),
            "sentence": f"Основная тема: {', '.join(words[:6])}. Эти слова часто встречаются в отзывах."
        })

    bertopic_df = pd.DataFrame(topic_rows)
    return bertopic_df

final_topics = topics_advanced(df)
# нужно подогать структуры таблиц финальных тем и в супабейзе одинаковыми



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [42]:
# Saving results to DB
def upload_to_db(final_topics, company_id):

    df = final_topics.copy()

    df = df.rename(columns={
        "sentence": "trend_text",
        "method": "source"
    })

    df["company_id"] = company_id
    df["trend_type"] = "general"

    if "relevance" not in df.columns:
        df["relevance"] = 0.8

    supabase = get_config()
    records = df.to_dict(orient="records")

    for r in records:
        data = {
            "company_id": int(r["company_id"]),
            "trend_text": r.get("trend_text", ""),
            "trend_type": r.get("trend_type", "general"),
            "source": r.get("source", "unknown"),
            "relevance": float(r.get("relevance", 0.8))
        }

        try:
            supabase.table("topic_summary").insert(data).execute()

            print("Inserted:", data["trend_text"][:60])

        except Exception as e:
            print("Error:", e, data)

upload_to_db(final_topics, 1)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Inserted: Основная тема: видео, не, вк, музыку, что, музыка. Эти слова
Inserted: Основная тема: реклама, рекламы, не, это, на, рекламу. Эти с
Inserted: Основная тема: kate, mobile, верните, вк, кейт, для. Эти сло
Inserted: Основная тема: дно, класс, зависает, вк, за, днище. Эти слов
Inserted: Основная тема: не, приложение, по, сообщения, вообще, приход
Inserted: Основная тема: не, сообщения, нет, приходят, звонки, вообще.
Inserted: Основная тема: приложение, приложения, то, не, что, очень. Э
Inserted: Основная тема: спасибо, нравится, мне, ок, супер, норм. Эти 
Inserted: Основная тема: аккаунт, могу, не, войти, номер, восстановить
Inserted: Основная тема: приложение, хорошее, очень, одном, удобно, эт
Inserted: Основная тема: каналы, каналов, чаты, там, раздел, они. Эти 
Inserted: Основная тема: хуже, каждым, работать, время, все, кругами. 
Inserted: Основная 

In [40]:
print(final_topics.columns)

Index(['method', 'topic_id', 'keywords', 'sentence', 'company_id'], dtype='object')
